In [1]:
import anndata
import sys
# sys.path.append("../../../../")
# sys.path.append("../../../../../")
sys.path.append("/archive/bioinformatics/DLLab/AixaAndrade/src/ARMED_genomics_git/utils")

from utils import read_adata
from splitter import DataSplitter
from model_train_utils import load_data
from config_split_paths import data_path,data2split_foldername,folder_splits
import glob
import anndata as ad

2024-08-29 12:04:29.732656: I tensorflow/stream_executor/platform/default/dso_loader.cc:48] Successfully opened dynamic library libcudart.so.10.1


In [2]:
# read paths
adata2split_path = data_path +"/"+folder_splits
print(adata2split_path)
splits_list = glob.glob(adata2split_path+"/*")
glob.glob(splits_list[0]+"/train/*")

/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits


['/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train/exprMatrix.npy',
 '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train/meta.csv',
 '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train/geneids.csv']

In [3]:
# get original donor data
adata2split_path_original = data_path + "/" + data2split_foldername
# X,var,obs = read_adata(adata2split_path_original)
X,var,obs = read_adata(adata2split_path_original, issparse=True)
adata = anndata.AnnData(X,obs=obs,var=var)

/project/bioinformatics/DLLab/shared/CondaEnvironments/Aixa_ARMED_2/lib/python3.8/site-packages/anndata/_core/anndata.py:119: ImplicitModificationWarning: Transforming to str index.
  warnings.warn("Transforming to str index.", ImplicitModificationWarning)


In [4]:
# Creating a dict with split paths and adata_splits
def get_splits_dict(splits_path,issparse=False):
    splits_list = glob.glob(splits_path+"/*")
    #print(splits_list)
    split_paths_dict = {}
    adata_splits_dict = {}
    for split in splits_list:
        split_name = split.split("/")[-1]
        split_paths_dict[split_name]={"train":split+"/train","val":split+"/val","test":split+"/test"}
        print(split_paths_dict[split_name])
        adata_splits_dict[split_name]=load_data(split_paths_dict[split_name], eval_test=True, scaling=None,issparse=issparse)
    return split_paths_dict,adata_splits_dict

In [5]:
def check_fold_leakage(adata_splits_dict):
    val_indices = []
    test_indices = []

    # Extracting indices for validation and test sets from each fold in the dictionary
    for split_key in adata_splits_dict.keys():
        fold_data = adata_splits_dict[split_key]
        val_indices.append(set(fold_data['val'].obs['original_index']))
        test_indices.append(set(fold_data['test'].obs['original_index']))

    # Checking for any overlaps (leakage) in validation and test sets
    num_folds = len(adata_splits_dict)
    overlap_detected = False
    for i in range(num_folds):
        for j in range(i + 1, num_folds):
            val_intersection = val_indices[i].intersection(val_indices[j])
            test_intersection = test_indices[i].intersection(test_indices[j])
            if val_intersection:
                print(f"Overlap detected in validation sets between folds {i} and {j}: {len(val_intersection)} common elements")
                overlap_detected = True
            if test_intersection:
                print(f"Overlap detected in test sets between folds {i} and {j}: {len(test_intersection)} common elements")
                overlap_detected = True

    if overlap_detected:
        return False
    else:
        print("No overlap detected in validation and test sets between folds.")
        return True


In [6]:
# Creating a dict with split paths and adata_splits
split_paths_dict,adata_splits_dict = get_splits_dict(data_path+"/"+folder_splits,issparse=True) 

{'train': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/train', 'val': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/val', 'test': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_1/test'}
X.shape before scaling (291680, 3000)
X.shape before scaling (97227, 3000)
X.shape before scaling (97227, 3000)
{'train': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_2/train', 'val': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/log_transformed_3000hvggenes/splits/split_2/val', 'test': '/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_data/Healthy_human_heart_data/l

In [7]:
data_path+folder_splits

'/archive/bioinformatics/DLLab/AixaAndrade/data/Genomic_data/heart_dataHealthy_human_heart_data/log_transformed_3000hvggenes/splits'

In [8]:


# Check if there is any overlap between 
overlap_check_result = check_fold_leakage(adata_splits_dict)

No overlap detected in validation and test sets between folds.


In [22]:
# get train data from split 1
split_to_check = "split_4"
adata_train = adata_splits_dict[split_to_check]["train"]
print("adata_train shape",adata_train.shape)
# read test data from split 1
adata_test = adata_splits_dict[split_to_check]["test"]
print("adata_test shape",adata_test.shape)
# read val data from split 1
adata_val = adata_splits_dict[split_to_check]["val"]
print("adata_val shape",adata_val.shape)

adata_train shape (291681, 3000)
adata_test shape (97227, 3000)
adata_val shape (97226, 3000)


In [23]:
# adata_test.obs

In [24]:
# adata.X.todense() #IT IS NOT BETWEEN [0,1]. The scaling is done when the data is loaded

In [25]:
# check that train, test and val should sum up total data
adata_train.shape[0]+adata_val.shape[0]+adata_test.shape[0]

486134

In [13]:
import numpy as np
# # check all donors list
adata.obs['batch'] = adata.obs["sampleID"].astype('category')
donor_list = list(np.unique(adata.obs["batch"]))
# Seen donor list
donor_seen_list = list(np.unique(adata_train.obs["batch"]))
print(donor_seen_list)

['H0015_LA_new', 'H0015_LV', 'H0015_RA', 'H0015_RV', 'H0015_apex', 'H0015_septum', 'H0020_LA_new', 'H0020_LV', 'H0020_RA', 'H0020_RV', 'H0020_apex', 'H0020_septum', 'H0025_LA', 'H0025_LV', 'H0025_RA', 'H0025_RV', 'H0025_apex', 'H0025_septum', 'H0026_LA', 'H0026_LV_V3', 'H0026_RA', 'H0026_RV', 'H0026_apex', 'H0026_septum2', 'H0035_LA', 'H0035_LV', 'H0035_RA', 'H0035_RV', 'H0035_apex', 'H0035_septum', 'H0037_Apex', 'H0037_LA_corr', 'H0037_LV', 'H0037_RA_corr', 'H0037_RV', 'H0037_septum', 'HCAHeart7606896', 'HCAHeart7656534', 'HCAHeart7656535', 'HCAHeart7656536', 'HCAHeart7656537', 'HCAHeart7656538', 'HCAHeart7656539', 'HCAHeart7664652', 'HCAHeart7664653', 'HCAHeart7664654', 'HCAHeart7698015', 'HCAHeart7698016', 'HCAHeart7698017', 'HCAHeart7702873', 'HCAHeart7702874', 'HCAHeart7702875', 'HCAHeart7702876', 'HCAHeart7702877', 'HCAHeart7702878', 'HCAHeart7702879', 'HCAHeart7702880', 'HCAHeart7702881', 'HCAHeart7702882', 'HCAHeart7728604', 'HCAHeart7728605', 'HCAHeart7728606', 'HCAHeart772860

In [14]:
# check if the stratification worked

# call the splitter stratification test
splitter = DataSplitter()
comp_df = splitter.check_stratification(adata, adata_train, adata_val, adata_test, ['batch', 'celltype'])
# add donor and celltype columns
comp_df["batch"] = [x.split("_")[0] for x in comp_df.index ]
comp_df["celltype"] = [x.split("_")[1] for x in comp_df.index ]


# Donor is in seen list
condition1 = comp_df["batch"].isin(donor_seen_list)
# Values equal to zero
condition2 = comp_df[['Train', 'Validation', 'Test']].eq(0).any(axis=1)
filtered_df = comp_df.loc[condition1 & condition2]
# We want to see the seen donors well distributed trough the train, test and validation datasets. No zero entries.


In [15]:
filtered_df

,Original,Train,Validation,Test,batch,celltype
HCAHeart7606896_Myeloid,0.000004,0.000007,0.000000,0.000000,HCAHeart7606896,Myeloid
HCAHeart7656534_Neuronal,0.000002,0.000004,0.000000,0.000000,HCAHeart7656534,Neuronal
HCAHeart7656535_Fibroblast,0.000004,0.000007,0.000000,0.000000,HCAHeart7656535,Fibroblast
HCAHeart7656535_Lymphoid,0.000004,0.000007,0.000000,0.000000,HCAHeart7656535,Lymphoid
HCAHeart7656537_Fibroblast,0.000004,0.000007,0.000000,0.000000,HCAHeart7656537,Fibroblast
...,...,...,...,...,...,...
HCAHeart8102867_Neuronal,0.000004,0.000007,0.000000,0.000000,HCAHeart8102867,Neuronal
HCAHeart8287123_Lymphoid,0.000004,0.000000,0.000011,0.000011,HCAHeart8287123,Lymphoid
HCAHeart8287124_Lymphoid,0.000004,0.000000,0.000011,0.000011,HCAHeart8287124,Lymphoid
HCAHeart8287124_Smooth_muscle_cells,0.000002,0.000000,0.000000,0.000011,HCAHeart8287124,Smooth


In [16]:
adata_val

AnnData object with n_obs × n_vars = 90159 × 2916
    obs: 'Unnamed: 0', 'Unnamed: 0.1', 'Age', 'AgeBin', 'DeathType', 'DonorID', 'Gender', 'Organ', 'Race', 'SampleType', 'Source', 'Tissue', 'TissueDetail', '_index', 'celltype', 'protocol', 'sampleID', 'n_genes', 'batch', 'original_index', 'combined_stratify_col'
    var: 'Unnamed: 0'

In [17]:
adata_train.obs["combined_stratify_col"]

0                          HCAHeart7905332_Lymphoid
1                H0025_LV_Ventricular_Cardiomyocyte
2         HCAHeart7888926_Ventricular_Cardiomyocyte
3         HCAHeart7656539_Ventricular_Cardiomyocyte
4                       HCAHeart7888927_Endothelial
                            ...                    
270472                    HCAHeart7702874_Pericytes
270473       H0035_septum_Ventricular_Cardiomyocyte
270474           H0020_RV_Ventricular_Cardiomyocyte
270475                   HCAHeart7702880_Fibroblast
270476                         H0035_apex_Pericytes
Name: combined_stratify_col, Length: 270477, dtype: object

In [18]:
adata_train.obs["combined_stratify_col"]

0                          HCAHeart7905332_Lymphoid
1                H0025_LV_Ventricular_Cardiomyocyte
2         HCAHeart7888926_Ventricular_Cardiomyocyte
3         HCAHeart7656539_Ventricular_Cardiomyocyte
4                       HCAHeart7888927_Endothelial
                            ...                    
270472                    HCAHeart7702874_Pericytes
270473       H0035_septum_Ventricular_Cardiomyocyte
270474           H0020_RV_Ventricular_Cardiomyocyte
270475                   HCAHeart7702880_Fibroblast
270476                         H0035_apex_Pericytes
Name: combined_stratify_col, Length: 270477, dtype: object

In [19]:
# from matplotlib import pyplot as plt
# index_val = adata_val.obs.loc[adata_val.obs['combined_stratify_col']=='5163_Neu-mat'].index
# print("index_val.shape",index_val.shape)
# plt.hist(adata_val.X[np.array(index_val.astype(int)),:])


In [20]:
adata_train.obs

,Unnamed: 0,Unnamed: 0.1,Age,AgeBin,DeathType,DonorID,Gender,Organ,Race,SampleType,...,Tissue,TissueDetail,_index,celltype,protocol,sampleID,n_genes,batch,original_index,combined_stratify_col
0,358595,389914,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,...,AX,apex,b'GAGTTTGCAACTGGTT-1-HCAHeart7905332',Lymphoid,10X3'V3,HCAHeart7905332,671,HCAHeart7905332,358595,HCAHeart7905332_Lymphoid
1,77562,83912,NaN,50-55,DBD(brain death),H3,Male,Heart,Asian,normal,...,LV,left ventricle,b'TTCGATTCACAGTATC-1-H0025_LV',Ventricular_Cardiomyocyte,10X3'V3,H0025_LV,551,H0025_LV,77562,H0025_LV_Ventricular_Cardiomyocyte
2,294516,320678,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,...,SP,septum,b'AGTGTCACAGCCACCA-1-HCAHeart7888926',Ventricular_Cardiomyocyte,10X3'V2,HCAHeart7888926,955,HCAHeart7888926,294516,HCAHeart7888926_Ventricular_Cardiomyocyte
3,181245,196524,NaN,60-65,DCD(circulatory death),D2,Male,Heart,Caucasian,normal,...,SP,septum,b'TCTGGAATCGAGCCCA-1-HCAHeart7656539',Ventricular_Cardiomyocyte,10X3'V2,HCAHeart7656539,533,HCAHeart7656539,181245,HCAHeart7656539_Ventricular_Cardiomyocyte
4,297185,324418,NaN,65-70,DCD(circulatory death),D6,Male,Heart,Caucasian,normal,...,AX,apex,b'ACGATACCATATGGTC-1-HCAHeart7888927',Endothelial,10X3'V2,HCAHeart7888927,1115,HCAHeart7888927,297185,HCAHeart7888927_Endothelial
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
270472,191335,207274,NaN,60-65,DCD(circulatory death),D2,Male,Heart,Caucasian,normal,...,RV,right ventricle,b'TGACTAGCAGACAAGC-1-HCAHeart7702874',Pericytes,10X3'V2,HCAHeart7702874,472,HCAHeart7702874,191335,HCAHeart7702874_Pericytes
270473,137337,149655,NaN,45-50,DBD(brain death),H7,Female,Heart,Caucasian,normal,...,SP,septum,b'CATCGCTGTCAACCTA-1-H0035_septum',Ventricular_Cardiomyocyte,10X3'V3,H0035_septum,2137,H0035_septum,137337,H0035_septum_Ventricular_Cardiomyocyte
270474,54886,58814,NaN,40-45,DBD(brain death),H6,Female,Heart,Caucasian,normal,...,RV,right ventricle,b'ACTTTCAGTCCTTTGC-1-H0020_RV',Ventricular_Cardiomyocyte,10X3'V3,H0020_RV,2333,H0020_RV,54886,H0020_RV_Ventricular_Cardiomyocyte
270475,207892,224923,NaN,60-65,DCD(circulatory death),D2,Male,Heart,Caucasian,normal,...,SP,septum,b'CACACTCCAAATACAG-1-HCAHeart7702880',Fibroblast,10X3'V2,HCAHeart7702880,575,HCAHeart7702880,207892,HCAHeart7702880_Fibroblast
